# **Les 1 - ML Pipeline**

Student: Jamey Verlinden

R nummer: r1042086

Richting: BADI 3

## Stap 1: Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import imblearn
import numpy as np

import seaborn as sns
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import accuracy_score
from imblearn.metrics import specificity_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

## Stap 2: Data

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

In [4]:
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [5]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


#### Categorical values

Imputate + Onehotencode

- pclass
- Cabin
- Embarked
- Gender

#### Numerical values

Imputate + scale

- Age
- Fare
- SibSp

## Stap 3: Preprocessing

In [6]:
Numeric_pipeline = Pipeline(steps=[
    ("Imputation", SimpleImputer(strategy="median")), # Why median? Fare can range to very high amounts, making averages less robust.
    ("Scaler", StandardScaler()) # Scale all numeric data. Best practice for ML because then larger numbers don't overpower the model.
])

Categorical_pipeline = Pipeline(steps=[
    ("Imputation", SimpleImputer(strategy="constant", fill_value="missing")),
    ("HotEncoder", OneHotEncoder(handle_unknown="ignore"))
])

Transformer = ColumnTransformer(transformers=[ # Columntransformer accepts 3 steps. That is why it is used over the "Pipeline", which only accepts 2.
    ("numeric transformation", Numeric_pipeline, ["Age", "Fare", "SibSp", "Parch"]),
    ("categorical transformation", Categorical_pipeline, ["Embarked", "Sex", "Pclass"])
])

## Stap 4: Splitsen in Test en Training

In [7]:
X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

## Stap 5: ML Pipeline

In [8]:
ML_pipeline = Pipeline(steps=[
    ("pre-processing", Transformer),
    ("machine model", RandomForestClassifier(random_state=3))
])

## Stap 6: Train het model

In [9]:
ML_pipeline.fit(X_train, y_train)

Pipeline(steps=[('pre-processing',
                 ColumnTransformer(transformers=[('numeric transformation',
                                                  Pipeline(steps=[('Imputation',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('Scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare', 'SibSp',
                                                   'Parch']),
                                                 ('categorical transformation',
                                                  Pipeline(steps=[('Imputation',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('HotEncoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Embarked', 'Sex',
                                                   'Pclass'])])),
                ('machine model', RandomForestClassifier(random_state=3))])

## Stap 7: Evaluatie

In [10]:
y_preds = ML_pipeline.predict(X_test)

accuracy_score = accuracy_score(y_preds, y_test)
precision = precision_score(y_preds, y_test)
F1_score = f1_score(y_preds, y_test)
recall = recall_score(y_preds, y_test)

In [11]:
print(f"Accuracy: {round(float(accuracy_score)*100, 2)}%")
print(f"Precision: {round(float(precision)*100, 2)}%")
print(f"F1_score: {round(float(F1_score)*100, 2)}%")
print(f"Recall: {round(float(recall)*100, 2)}%") # => Important to keep this high. Because it's better to predict someone has died when they did not, rather then predicting they survived when they actually perished.

Accuracy: 77.24%
Precision: 64.91%
F1_score: 70.81%
Recall: 77.89%


## Stap 8: Predictie

In [12]:
# 'Pclass': 3, 'Sex': 'male', 'Age': 25, 'SibSp': 0, 'Parch': 0, 'Fare': 7.5, 'Embarked': 'S'

predict_this = pd.DataFrame({
    "Pclass": [3],
    "Sex": ["male"],
    "Age": [25],
    "SibSp": [0],
    "Parch": [0],
    "Fare": [7.5],
    "Embarked": ["S"]
})

Prediction = ML_pipeline.predict(predict_this)
print(Prediction)

[0]


Unfortunately, he did not survive.